In [ ]:
@file:DependsOn("org.postgresql:postgresql:42.7.4")
@file:OptIn(ExperimentalUuidApi::class)

import kotlin.uuid.ExperimentalUuidApi

In [ ]:
%useLatestDescriptors
%use dataframe
%use lets-plot
//%use kandy

In [ ]:
import bps.jdbc.getConfigFromResource

val (jdbcConfig, hikariConfig) = getConfigFromResource("/home/benji/.config/bps-budget-server/budget-server.yml")

In [ ]:
hikariConfig

In [ ]:
import bps.jdbc.configureDataSource

val dataSource = configureDataSource(jdbcConfig, hikariConfig)

In [ ]:
import java.util.UUID

val budgetAccessId = "7e8366d3-6710-4342-a3ea-26282ddb65f6"
val budgetName = "Benji's Budget"
val userId = UUID.fromString("c2d15fe2-3dfe-4517-b9ce-ab1b55f6b320")

In [ ]:
import bps.jdbc.JdbcFixture.Companion.transactOrThrow
import java.sql.PreparedStatement
import java.sql.ResultSet

val basicData = DataFrame.readSqlQuery(
    dataSource = dataSource,
//                                join users u on u.id = ba.user_id
    sqlQuery =     """
                            select b.general_account_name, b.id as budget_id, ba.time_zone, ba.budget_name, ba.analytics_start, a.id as general_account_id
                            from budgets b
                                join budget_access ba on b.id = ba.budget_id
                                join accounts a
                                    on a.name = b.general_account_name
                                    and a.type = 'category'
                                    and a.budget_id = b.id
                            where ba.budget_name = ?
                                and ba.user_id = ?
                        """.trimIndent(),
    limit = 1000,
) { statement: PreparedStatement ->
    statement.setString(1, budgetName)
    statement.setObject(2, userId, java.sql.Types.OTHER)
}

In [ ]:
basicData

In [ ]:
@file:OptIn(ExperimentalUuidApi::class, ExperimentalTime::class)

import bps.jdbc.JdbcFixture
import java.math.BigDecimal
import kotlin.time.ExperimentalTime
import kotlin.time.toKotlinInstant
import kotlin.uuid.ExperimentalUuidApi
import kotlin.uuid.Uuid

val sqlQuery = """
                |select t.timestamp_utc, i.amount, a.name, t.description
                |from transaction_items i
                |join transactions t
                |    on i.transaction_id = t.id
                |    and i.budget_id = t.budget_id
                |join accounts a
                |    on i.account_id = a.id
                |    and i.budget_id = a.budget_id
                |where t.type = 'income'
                |  and i.amount > 0
                |  and a.type = 'real'
                |  and t.timestamp_utc >= ?
                |  and t.timestamp_utc < ?
                |  and i.budget_id = ?
                |order by t.timestamp_utc asc
            """.trimMargin()
val startKotlinInstant = basicData.analytics_start.values.first().toInstant().toKotlinInstant()
val endKotlinInstant: kotlin.time.Instant = kotlin.time.Clock.System.now() - 30.days
var income =
    DataFrame
        .readSqlQuery(dataSource, sqlQuery) {
            with(JdbcFixture) {
                it.setInstant(1, startKotlinInstant)
                it.setInstant(2, endKotlinInstant)
                it.setUuid(3, Uuid.parse(basicData.budget_id.values.first().toString()))
            }
        }
income

In [ ]:
income = income
    .convert { amount }
    .to<Double>()
income.schema()


In [ ]:
income


In [ ]:
income
    .filter {timestamp_utc.toLocalDateTime().year == 2025}
    .groupBy("name", "description")
    .aggregate { sum { amount } into "amount" }
